# 04 — Baseline and modeling

**Task:** unsupervised detection of unusual one-minute compressor windows, not supervised prediction of future failures. Train only through March 2020, select the operational alert threshold on April–May validation, and reserve June–August as untouched test. Compare a robust-distance baseline and Isolation Forest. Incident overlap is descriptive, not a supervised accuracy metric.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import RAW, PROCESSED, MODELS, ANALOG, DIGITAL, FAILURES

import joblib
from src.models.train import fit_detector,fit_statistical_baseline,baseline_scores,save_model
from src.models.predict import anomaly_scores
from src.evaluation.evaluate import summarize_alerts,event_coverage
train=pd.read_pickle(PROCESSED/'train_features.pkl');val=pd.read_pickle(PROCESSED/'val_features.pkl')
cols=[c for c in train if c!='timestamp']; X=train[cols];V=val[cols]
model=fit_detector(X); baseline=fit_statistical_baseline(X)
s_if=anomaly_scores(model,V);s_base=baseline_scores(V,baseline)

## Threshold policy and comparison

Use a **fixed validation alert budget of 1% of windows** for both methods. This is an illustrative operational constraint, not a claim that 1% are actual failures. The same quantile policy gives similar alert counts, so selection by overlap with documented incidents is exploratory and potentially optimistic. We choose Isolation Forest as the predeclared primary model; the baseline is a diagnostic comparator. The test remains untouched.

In [ ]:
results=[]
for name,s in [('robust_baseline',s_base),('isolation_forest',s_if)]:
    threshold=float(np.quantile(s,.99))
    results.append({'method':name,'threshold':threshold,**summarize_alerts(val.timestamp,s,threshold)})
display(pd.DataFrame(results))
threshold=float(np.quantile(s_if,.99))
display(event_coverage(val.timestamp,s_if,threshold))
save_model(model,MODELS/'isolation_forest.joblib')
joblib.dump(cols,MODELS/'feature_columns.joblib')
joblib.dump({'threshold':threshold,'validation_alert_quantile':.99,'seed':42},MODELS/'decision_policy.joblib')
pd.DataFrame(results).to_csv(PROCESSED/'validation_comparison.csv',index=False)

## Selection and reproducibility

Isolation Forest is the primary model chosen a priori, with deterministic seed 42 and 150 trees. Validation determines only its decision threshold. The final model and feature column order are saved for independent inference. Do not interpret the validation incident overlap as an unbiased estimate of predictive performance.